In [83]:
print("NYC PAYROLL ETL PIPELINE")

NYC PAYROLL ETL PIPELINE


In [84]:
from google.colab import files
import pandas as pd
import uuid
uploaded = files.upload()

Saving Citywide_Payroll_Data_(Fiscal_Year)_20260518.csv to Citywide_Payroll_Data_(Fiscal_Year)_20260518 (4).csv


In [85]:
df = pd.read_csv("Citywide_Payroll_Data_(Fiscal_Year)_20260518.csv", low_memory=False)

print("Loaded:", df.shape)

Loaded: (550219, 17)


In [86]:
df = df.sample(n=50000, random_state=42)

print("Sampled:", df.shape)

Sampled: (50000, 17)


In [87]:
def anonymize_person(first, middle, last):
    full_name = (
        str(first if pd.notnull(first) else "").strip().lower() +
        str(middle if pd.notnull(middle) else "").strip().lower() +
        str(last if pd.notnull(last) else "").strip().lower()
    )
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, full_name))

In [88]:
df["employee_id"] = df.apply(
    lambda row: anonymize_person(
        row["First Name"],
        row["Mid Init"],
        row["Last Name"]
    ),
    axis=1
)

In [89]:
df = df.drop(columns=["First Name", "Mid Init", "Last Name"])

In [90]:
numeric_cols = [
    "Base Salary",
    "Regular Hours",
    "Regular Gross Paid",
    "OT Hours",
    "Total OT Paid",
    "Total Other Pay"
]

df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [91]:
df[numeric_cols] = df[numeric_cols].fillna(0)

In [92]:
print(df.shape)
print(df.head())
print(df.isnull().sum())
print(df.dtypes)

(50000, 15)
        Fiscal Year  Payroll Number                     Agency Name  \
354868         2025             747  DEPT OF ED PER SESSION TEACHER   
504215         2025              56               POLICE DEPARTMENT   
218661         2025             742          DEPT OF ED PEDAGOGICAL   
285401         2025             742          DEPT OF ED PEDAGOGICAL   
373386         2025             747  DEPT OF ED PER SESSION TEACHER   

       Agency Start Date Work Location Borough             Title Description  \
354868        10/12/2012             MANHATTAN          TEACHER- PER SESSION   
504215        07/09/2013              RICHMOND                POLICE OFFICER   
218661        10/05/2000             MANHATTAN                       TEACHER   
285401        09/05/2017             MANHATTAN                       TEACHER   
373386        06/01/2004             MANHATTAN  SCHOOL SECRETARY PER SESSION   

       Leave Status as of June 30  Base Salary  Pay Basis  Regular Hours  \
3548

In [93]:
df["Base Salary"].describe()

,Base Salary
count,50000.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


**STAR SCHEMA**

In [94]:
fact_payroll = df[[
    "employee_id",
    "Fiscal Year",
    "Payroll Number",
    "Agency Name",
    "Title Description",
    "Base Salary",
    "Regular Hours",
    "Regular Gross Paid",
    "OT Hours",
    "Total OT Paid",
    "Total Other Pay"
]].copy()

print(fact_payroll.head())
print(fact_payroll.shape)

                                 employee_id  Fiscal Year  Payroll Number  \
354868  442419c9-ff75-5b33-aa8f-4383e661a4c3         2025             747   
504215  7b3746f7-1860-52a9-ac45-715521993846         2025              56   
218661  a7ee6f1e-2b38-5fc8-8477-453429eaf05b         2025             742   
285401  9b181dd8-af1a-53bd-b810-d92af88eaee6         2025             742   
373386  230b7753-cd05-5dec-80ab-f22e5277279c         2025             747   

                           Agency Name             Title Description  \
354868  DEPT OF ED PER SESSION TEACHER          TEACHER- PER SESSION   
504215               POLICE DEPARTMENT                POLICE OFFICER   
218661          DEPT OF ED PEDAGOGICAL                       TEACHER   
285401          DEPT OF ED PEDAGOGICAL                       TEACHER   
373386  DEPT OF ED PER SESSION TEACHER  SCHOOL SECRETARY PER SESSION   

        Base Salary  Regular Hours  Regular Gross Paid  OT Hours  \
354868          0.0            0.0  

In [95]:
dim_employee = df[[
    "employee_id",
    "Work Location Borough"
]].drop_duplicates()

print(dim_employee.head())

                                 employee_id Work Location Borough
354868  442419c9-ff75-5b33-aa8f-4383e661a4c3             MANHATTAN
504215  7b3746f7-1860-52a9-ac45-715521993846              RICHMOND
218661  a7ee6f1e-2b38-5fc8-8477-453429eaf05b             MANHATTAN
285401  9b181dd8-af1a-53bd-b810-d92af88eaee6             MANHATTAN
373386  230b7753-cd05-5dec-80ab-f22e5277279c             MANHATTAN


In [96]:
dim_department = df[[
    "Payroll Number",
    "Agency Name"
]].drop_duplicates()

print(dim_department.head())

        Payroll Number                     Agency Name
354868             747  DEPT OF ED PER SESSION TEACHER
504215              56               POLICE DEPARTMENT
218661             742          DEPT OF ED PEDAGOGICAL
89206              740   DEPARTMENT OF EDUCATION ADMIN
154518             744   DEPT OF ED PARA PROFESSIONALS


In [97]:
dim_role = df[[
    "Title Description",
    "Pay Basis"
]].drop_duplicates()

print(dim_role.head())

                   Title Description  Pay Basis
354868          TEACHER- PER SESSION    per Day
504215                POLICE OFFICER  per Annum
218661                       TEACHER  per Annum
373386  SCHOOL SECRETARY PER SESSION    per Day
299282           ASSISTANT PRINCIPAL  per Annum


In [98]:
dim_time = df[[
    "Fiscal Year"
]].drop_duplicates()

print(dim_time.head())

        Fiscal Year
354868         2025


In [99]:
print("FACT:", fact_payroll.shape)
print("EMP:", dim_employee.shape)
print("DEPT:", dim_department.shape)
print("ROLE:", dim_role.shape)
print("TIME:", dim_time.shape)

FACT: (50000, 11)
EMP: (49064, 2)
DEPT: (108, 2)
ROLE: (987, 2)
TIME: (1, 1)
